# 将 dataset 短句填入 sentences.json（含拼音与分词数）

1. 从 `dataset.txt` 或 `dataset` 按行读取短句，清理后追加到 `app/.../sentences.json`。  
2. **为每条句子准备拼音和分词数**：用 `pypinyin` 生成带声调拼音，用 `jieba` 分词得到 `word_count`，并写回 JSON。

In [ ]:
import json
import re
from pathlib import Path

# 项目根目录：在 Jupyter 中一般为当前工作目录，请把笔记本放在 Chinese_game 下或修改 BASE
BASE = Path('.').resolve()
if 'Chinese_game' not in str(BASE):
    BASE = BASE / 'Chinese_game'  # 若在上级目录打开笔记本

DATASET_FILE = BASE / 'dataset.txt'   # 若无 dataset.txt 则下面会改用 dataset
SENTENCES_JSON = BASE / 'app' / 'src' / 'main' / 'assets' / 'json' / 'sentences.json'

print('Dataset:', DATASET_FILE, 'exists:', DATASET_FILE.exists())
print('Sentences:', SENTENCES_JSON, 'exists:', SENTENCES_JSON.exists())

Dataset: D:\OneDrive - Queen Mary, University of London\Desktop\code\AndroidCode\Chinese_game\dataset.txt exists: False
Sentences: D:\OneDrive - Queen Mary, University of London\Desktop\code\AndroidCode\Chinese_game\app\src\main\assets\json\sentences.json exists: True


In [23]:
# 若未安装，取消下一行注释并运行一次
# !pip install pypinyin jieba

from pypinyin import pinyin, Style
import jieba

def sentence_to_pinyin(s: str) -> str:
    """将句子转为带声调、音节间空格的拼音（与 app 中 PinyinUtils 格式一致）。"""
    if not s or not s.strip():
        return ""
    parts = []
    for c in s.strip():
        py = pinyin(c, style=Style.TONE, neutral_tone_with_five=True)
        if py and py[0]:
            parts.append(py[0][0])
        else:
            parts.append(c)
    return " ".join(parts)

def get_segment_count(s: str) -> int:
    """分词并返回词数（分词数）。"""
    if not s or not s.strip():
        return 0
    return len(jieba.lcut(s.strip()))

# 试一条
t = "我喜欢在北京学习中文"
print("拼音:", sentence_to_pinyin(t))
print("分词数:", get_segment_count(t))

Building prefix dict from the default dictionary ...


拼音: wǒ xǐ huān zài běi jīng xué xí zhōng wén


Dumping model to file cache C:\Windows\TEMP\jieba.cache
Loading model cost 0.892 seconds.
Prefix dict has been built successfully.


分词数: 6


In [25]:
def clean_line(line: str) -> str:
    """去掉首尾空白、引号、项目符号。"""
    s = line.strip()
    s = re.sub(r"^[\s\"\\'\u201c\u201d\u2018\u2019\uff07\u00b7\u2022\s]+", "", s)
    s = re.sub(r"[\s\"\\'\u201c\u201d\u2018\u2019\uff07\s]+$", "", s)
    return s.strip()

# 若 dataset.txt 不存在则尝试 dataset
input_path = DATASET_FILE if DATASET_FILE.exists() else BASE / 'dataset'
with open(input_path, 'r', encoding='utf-8', newline=None) as f:
    text = f.read()

lines = [clean_line(ln) for ln in re.split(r"[\r\n]+", text)]
lines = [ln for ln in lines if ln]
seen = set()
new_sentences = []
for s in lines:
    if s not in seen:
        seen.add(s)
        new_sentences.append(s)

print(f'从 dataset 得到 {len(new_sentences)} 条不重复短句（共 {len(lines)} 行）')

从 dataset 得到 470 条不重复短句（共 480 行）


In [27]:
with open(SENTENCES_JSON, 'r', encoding='utf-8') as f:
    sentences = json.load(f)

existing_sentences = {item['sentence'] for item in sentences}
max_id = max(item['id'] for item in sentences) if sentences else 0

to_add = [s for s in new_sentences if s not in existing_sentences]
for i, sentence_text in enumerate(to_add, start=1):
    sentences.append({
        "id": max_id + i,
        "sentence": sentence_text,
        "pinyin": sentence_to_pinyin(sentence_text),
        "difficulty": "MEDIUM",
        "category": "dataset",
        "word_count": get_segment_count(sentence_text)
    })

print(f'原有 {len(sentences) - len(to_add)} 条，新增 {len(to_add)} 条，合计 {len(sentences)} 条')

原有 474 条，新增 0 条，合计 474 条


## 为 sentences.json 中每条补充拼音与分词数

对当前 JSON 中**所有**句子统一生成 `pinyin`（带声调、空格分隔）和 `word_count`（jieba 分词数），并写回。可单独运行本单元以刷新已有数据。

In [30]:
with open(SENTENCES_JSON, 'r', encoding='utf-8') as f:
    sentences = json.load(f)

for item in sentences:
    s = item.get("sentence", "")
    item["pinyin"] = sentence_to_pinyin(s)
    item["word_count"] = get_segment_count(s)

with open(SENTENCES_JSON, 'w', encoding='utf-8') as f:
    json.dump(sentences, f, ensure_ascii=False, indent=2)

print(f'已为 {len(sentences)} 条句子写入拼音与分词数，已保存至 {SENTENCES_JSON}')

已为 474 条句子写入拼音与分词数，已保存至 D:\OneDrive - Queen Mary, University of London\Desktop\code\AndroidCode\Chinese_game\app\src\main\assets\json\sentences.json


In [32]:
with open(SENTENCES_JSON, 'w', encoding='utf-8') as f:
    json.dump(sentences, f, ensure_ascii=False, indent=2)

print('已写回:', SENTENCES_JSON)

已写回: D:\OneDrive - Queen Mary, University of London\Desktop\code\AndroidCode\Chinese_game\app\src\main\assets\json\sentences.json
